In [1]:
import carla
import time
from math import *
client = carla.Client('localhost', 2000)
world = client.get_world()
map = world.get_map()
spawn_point = map.get_spawn_points()[0]

In [2]:
spectator = world.get_spectator()
spectator.set_transform(spawn_point)

In [3]:

vehicle_blueprint= world.get_blueprint_library().filter("*.nissan.*")

vehicle = world.try_spawn_actor(vehicle_blueprint[0], spawn_point)

if vehicle == None:
    
    vehicle = world.get_actors().filter("vehicle.*")
    for v in vehicle:
        v.destroy()

world.wait_for_tick()

vehicle_position = vehicle.get_transform()
print(vehicle_position)



Transform(Location(x=-7.530000, y=142.190002, z=0.484628), Rotation(pitch=0.000000, yaw=89.999954, roll=0.000000))


In [113]:
print(vehicle.get_transform())

Transform(Location(x=7.433792, y=139.537354, z=0.310294), Rotation(pitch=-0.367949, yaw=-27.686451, roll=-1.209351))


In [138]:
vehicle.set_transform(spawn_point)

In [ ]:
vehicle.set_transform()

In [47]:
def steering_angle(Current_position : carla.Transform, Goal : carla.Transform):
    relative_vector = carla.Vector2D(Goal.location.x - Current_position.location.x, Goal.location.y - Current_position.location.y)
    Rotation_matrix = [[cos(radians(-Current_position.rotation.yaw)), -sin(radians(-Current_position.rotation.yaw))],
                       [sin(radians(-Current_position.rotation.yaw)), cos(radians(-Current_position.rotation.yaw))]]
    rotated_relative_vector = carla.Vector2D(Rotation_matrix[0][0] * relative_vector.x + Rotation_matrix[0][1] * relative_vector.y,
                                            Rotation_matrix[1][0] * relative_vector.x + Rotation_matrix[1][1] * relative_vector.y)
    print(f"rotated_relative_vector: {rotated_relative_vector}")
    Ld2 = rotated_relative_vector.squared_length()
    curvature = (2 * rotated_relative_vector.y) / Ld2
    print(f"rotated_relative_vector.y: {rotated_relative_vector.y}")
    print(f"curvature: {curvature}")
    angle = degrees(atan(2.5 * curvature))

    return angle

def Pure_pursuit(vehicle : carla.Vehicle, Goal : carla.Transform):

    Next_goal = Goal
    Current_position = vehicle.get_transform()

    Angle = steering_angle(Current_position, Next_goal)
    
    return Angle

In [54]:
vehicle.set_transform(spawn_point)

vehicle.apply_control(carla.VehicleControl(throttle=0.0, steer=0))
world.wait_for_tick()
print(vehicle.get_wheel_steer_angle(carla.VehicleWheelLocation.FL_Wheel))
print(vehicle.get_wheel_steer_angle(carla.VehicleWheelLocation.FR_Wheel))

0.0
0.0


In [ ]:
new_point = map.get_waypoint(spawn_point.location).next(10)[0]

 
world.debug.draw_point(new_point.transform.location + carla.Vector3D(0,0,1), size=0.2, color=carla.Color(0,255, 0), life_time=10.0)

while True:
    dx = new_point.transform.location.x - vehicle.get_transform().location.x
    dy = new_point.transform.location.y - vehicle.get_transform().location.y
    dist = (dx*dx + dy*dy) ** 0.5
    if dist < 7:
        world.debug.draw_point(new_point.transform.location + carla.Vector3D(0,0,1), size=0.2, color=carla.Color(255, 0, 0), life_time=10.0)
        new_point = new_point.next(10)[0]
        world.debug.draw_point(new_point.transform.location + carla.Vector3D(0,0,1), size=0.2, color=carla.Color(0,255, 0), life_time=10.0)
    Steerangle = Pure_pursuit(vehicle, new_point.transform)
    print(f"Vehicle Location: {vehicle.get_transform().location}")
    print(f"Goal_Point: {new_point.transform.location}")
    print(f"required Steering wheel angle: {Steerangle}")
    
    Angle = Steerangle / 70 
    # if Steerangle > 0:
        # Angle *= -1
    vehicle.apply_control(carla.VehicleControl(throttle=0.625, steer=Angle))
    world.wait_for_tick()



rotated_relative_vector: Vector2D(x=8.199355, y=-1.500623)
rotated_relative_vector.y: -1.5006225109100342
curvature: -0.0431950083847111
Vehicle Location: Location(x=-8.832125, y=143.972015, z=0.252870)
Goal_Point: Location(x=-7.437860, y=152.190125, z=0.000000)
required Steering wheel angle: -6.163345689227823
rotated_relative_vector: Vector2D(x=8.199382, y=-1.500642)
rotated_relative_vector.y: -1.5006422996520996
curvature: -0.04319526495648788
Vehicle Location: Location(x=-8.832127, y=143.971985, z=0.252675)
Goal_Point: Location(x=-7.437860, y=152.190125, z=0.000000)
required Steering wheel angle: -6.163382016798317
rotated_relative_vector: Vector2D(x=8.198668, y=-1.500248)
rotated_relative_vector.y: -1.500247597694397
curvature: -0.04319192822499464
Vehicle Location: Location(x=-8.832084, y=143.972763, z=0.252662)
Goal_Point: Location(x=-7.437860, y=152.190125, z=0.000000)
required Steering wheel angle: -6.162909574155962
rotated_relative_vector: Vector2D(x=8.195797, y=-1.499709)
r

KeyboardInterrupt: 

In [ ]:
vehicle.destroy()


True

AttributeError: 'DebugHelper' object has no attribute 'clear'

In [128]:
Vehicles = world.get_actors().filter("vehicle.*")
for v in Vehicles:
    v.destroy()

In [132]:
topology = world.get_map().get_topology()
for segment in topology:
    Location = carla.Location(segment[0].transform.location.x, segment[0].transform.location.y, segment[0].transform.location.z + 0.1)
    world.debug.draw_point(Location, size=0.3, color=carla.Color(255, 0, 0))
    time.sleep(5)

KeyboardInterrupt: 